In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q1_data.csv")

df_food = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: Write your code here:
df_food.dropna(subset=['Order_ID'])


In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_food.isnull().sum())

df_food.isnull().fillna('none')

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df_food.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df_food.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")



In [ ]:
cols = ['Distance_km', 'Order_ID', 'Preparation_Time_min', 'Traffic_Level','Delivery_Time','Weather', 'Time_of_Day', 'Vehicle_Type']
df_clean = df_food[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Preparation_Time_min','Traffic_Level','Distance_km'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Task 4: Write your code here:
categorical_cols = df_food.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


encoder = LabelEncoder()
for col in categorical_cols:
  df_clean[col] = encoder.fit_transform(df_clean[col]).copy()

print("Original:", categorical_cols)
print("Encoded:\n", encoded)

In [ ]:
df_clean.info()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Task 5: Write your code here:

X = df_clean[categorical_cols]
y = df_clean['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Order ID in train: {y_train.sum()}, in test: {y_test.sum()}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:

In [ ]:
from sklearn.ensemble import RandomForestRegressor


# Task 1: Write your code here:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
from sklearn.model_selection import train_test_split, KFold

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Task 2,3,4,5: Write your code here:
y_pred = model.predict(X_test_scaled)


mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': categorical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: